# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RAJkumar5736602/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Ranking via binary classification:** predict P(page is declining) and sort by it. A ranked queue is what the editor consumes, so ranking metrics matter most.

In [1]:
import sys, pathlib
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p/'data/raw').exists())
sys.path.insert(0, str(root/'work/scripts'))
from wk import *
import pandas as pd, numpy as np
df = load()
print(df.y.value_counts())

y
1    16262
0    13738
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `is_declining = trend_direction == 'down'` — a defined rule on observed impressions (last 30d vs previous 30d, >10% drop). It is a proxy for 'needs refresh', not a ground-truth editorial judgement.

In [2]:
print(pd.crosstab(df.trend_direction, df.y))

y                   0      1
trend_direction             
down                0  16262
flat             1152      0
new              2236      0
stable           5962      0
up               4388      0


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric:** Precision@50 on held-out clients (editor capacity ≈ 50 reviews), always reported next to the base rate; ROC AUC as the threshold-free discrimination check.

In [3]:
print('base rate (any random 50 picks would hit this share):', round(df.y.mean(),3))

base rate (any random 50 picks would hit this share): 0.542


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page (content_id), unique.

In [4]:
print('unique content_id:', df.content_id.is_unique)
df.head(3)[['content_id','client_id','impressions_90d','avg_position','content_age_days','trend_direction']]

unique content_id: True


,content_id,client_id,impressions_90d,avg_position,content_age_days,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,10.6,187,down
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,445,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,141,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Decline depends on interacting signals (age × visibility × position × length) across 32 clients with different baselines; a single if-statement cannot weigh these. The check below shows no single feature separates the classes well.

In [5]:
from sklearn.metrics import roc_auc_score
for c in ['content_age_days','impressions_90d','avg_position','word_count','days_since_last_update']:
    v = pd.to_numeric(df[c], errors='coerce').fillna(0)
    print(c, 'single-feature AUC =', round(roc_auc_score(df.y, v),3))

content_age_days single-feature AUC = 0.409
impressions_90d single-feature AUC = 0.584
avg_position single-feature AUC = 0.53
word_count single-feature AUC = 0.565
days_since_last_update single-feature AUC = 0.527


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.